# LAT 1 transition: Deep TICA with pairwise distances


In [2]:
from deep_cartograph.tools import traj_projection, compute_features
from deep_cartograph.deep_carto import deep_cartograph 
from deep_cartograph.modules.common import check_data
import importlib.resources as resources
from deep_cartograph import data

from IPython.display import display, HTML
from typing import List, Optional
import matplotlib.pyplot as plt
from decimal import Decimal
from pathlib import Path
import pandas as pd
import numpy as np
import shutil
import logging
import yaml
import time
import os

# Get the path to the data
data_folder = resources.files(data)

# Set logging level
logging.basicConfig(level=logging.INFO)

def project_evaluation_data(evaluation_traj_data: str,
                            evaluation_top_data: str,
                            output_folder,
                            model_name) -> List[str]:
    """ 
    Project evaluation data and return the paths to the projected data files.
    """
    
    trajectories, topologies = check_data(evaluation_traj_data, evaluation_top_data)
    trajectory_names = [Path(traj).stem for traj in trajectories]
    
    # Read configuration used to compute features
    compute_features_config = os.path.join(output_folder, 'compute_features', 'configuration.yml')
    with open(compute_features_config) as config_file:
        configuration = yaml.load(config_file, Loader = yaml.FullLoader)
    reference_topology = os.path.join(output_folder, 'compute_features', 'ref_topology.pdb')

    # Compute features
    args = {
        'configuration': configuration, 
        'trajectories': trajectories, 
        'topologies': topologies, 
        'reference_topology': reference_topology,
        'output_folder': os.path.join(output_folder, 'compute_features_eval')
    }
    traj_colvars_paths = compute_features(**args)
    
    # Read configuration to project trajectories
    projection_config = os.path.join(output_folder, 'traj_projection', 'configuration.yml')
    with open(projection_config) as config_file:
        configuration = yaml.load(config_file, Loader = yaml.FullLoader)
    
    # Find model path
    model_path = os.path.join(output_folder, 'train_colvars', model_name, 'model.zip')
    
    # Project evaluation trajectories
    args = { 
        'configuration' : configuration,
        'colvars_paths': traj_colvars_paths,
        'topologies': topologies,
        'trajectory_names': trajectory_names,
        'model_paths': [model_path],
        'model_traj_paths': None,
        'output_folder': os.path.join(output_folder, 'traj_projection_eval')
    }
    proj_eval_data = traj_projection(**args)
    
    return proj_eval_data.get(model_name, {}).get('traj_paths', [])

def create_time_evolution(proj_training_data_path: str, 
                          proj_evaluation_data_paths: List[str], 
                          output_path: str):
    """
    Create a time evolution plot with the evaluation data represented as a 
    min-max shadow (error/generalization spread) behind the training data.
    """

    # Read projected training data
    proj_data = pd.read_csv(proj_training_data_path)
    
    proj_eval_data = []
    if len(proj_evaluation_data_paths) > 0:
        logging.info(f"Processing {len(proj_evaluation_data_paths)} evaluation datasets for error estimation.")
        # Read projected evaluation data
        for eval_path in proj_evaluation_data_paths:
            proj_eval_data.append(pd.read_csv(eval_path))

    # Create time arrays
    # We do not assume that the evaluation data has the same time steps as the training data.
    train_time_array = np.arange(len(proj_data))
    eval_time_array = np.linspace(0, len(proj_data)-1, num=len(proj_eval_data[0])) if len(proj_eval_data) > 0 else None
    
    # Find number of components
    n_components = proj_data.shape[1]

    # --- Custom Font Sizes ---
    LABEL_SIZE = 18
    TITLE_SIZE = 18
    TICK_SIZE = 18
    LEGEND_SIZE = 18

    # Create plot
    plt.figure(figsize=(10, 8))
    
    # Use a colormap to ensure the line and its shadow share the same color
    colors = plt.cm.get_cmap('tab10', n_components)

    for i in range(n_components):
        # Get specific color for this CV
        c = colors(i)
        
        # 1. Plot the Training Data (The main signal)
        plt.plot(train_time_array, proj_data.iloc[:, i], 
                 label=f'Training data - CV {i+1}', 
                 color=c, 
                 linewidth=2,
                 zorder=2) # zorder=2 ensures line is on top of shadow
        
        # 2. Calculate and Plot the Shadow (The evaluation spread)
        if len(proj_evaluation_data_paths) > 0:
            # Extract the i-th column from all evaluation dataframes
            # resulting list shape: [ (N_steps,), (N_steps,), ... ]
            eval_values_list = [df.iloc[:, i].values for df in proj_eval_data]
            
            # Stack them to get a shape of (N_eval_datasets, N_steps)
            # This allows us to compute stats column-wise
            eval_matrix = np.vstack(eval_values_list)
            
            # Calculate Min and Max across the datasets (axis=0)
            eval_min = np.min(eval_matrix, axis=0)
            eval_max = np.max(eval_matrix, axis=0)
            
            # Create the shadow
            # We use the same 'train_time_array' for x-axis as per "training + noise" assumption
            plt.fill_between(eval_time_array, eval_min, eval_max, 
                             color=c, 
                             alpha=0.3, # Transparency
                             label=f'Evaluation data - CV {i+1}',
                             zorder=1) # zorder=1 ensures shadow is behind line

    # Apply Font Sizes
    plt.xlabel('Time step', fontsize=LABEL_SIZE)
    plt.ylabel('Projected CV', fontsize=LABEL_SIZE)
    plt.title('Time Evolution: Generalization Capability', fontsize=TITLE_SIZE)
    
    # Apply tick label sizes
    plt.xticks(fontsize=TICK_SIZE)
    plt.yticks(fontsize=TICK_SIZE)
    
    # Apply legend size
    plt.legend(fontsize=LEGEND_SIZE)
    
    plt.grid(True, alpha=0.5)
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()

def show_results(output_folder: str, 
                 model_name: str, 
                 system: str, 
                 evaluation_traj_data: Optional[str] = None, 
                 evaluation_top_data: Optional[str] = None):
    """
    Show the results for a specific model trained with deep cartograph

    Inputs
    ------

        output_folder   (str):          path to the output folder
        model_name      (str):          name of the model
    """

    def show_score(score_path):
        """
        Print score path in a nice format 

        Input
        -----

            score_path: path to the score file
        """

        # Read score
        with open(score_path, 'r') as file:
            score = file.read()

        # Print score in scientific notation
        print(f"Final model score: {Decimal(score):.4E}")

    def show_eigenvalues(eig_path):
        """
        Print eigenvalues in a nice format

        Input
        -----

            eig_path: path to the eigenvalues file
        """

        # Read eigenvalues
        with open(eig_path, 'r') as file:
            eigenvalues = file.readlines()

        # Print eigenvalues
        for i, eig in enumerate(eigenvalues):
            print(f"Eigenvalue {i+1}: {Decimal(eig):.4E}")

    # Check necessary folders
    print(f"Output folder: {output_folder}")
    train_cv_folder = os.path.join(output_folder, 'train_colvars')
    if not os.path.exists(train_cv_folder):
        print("Training folder not found")
        return
    traj_projection_folder = os.path.join(output_folder, 'traj_projection')
    if not os.path.exists(traj_projection_folder):
        print("Trajectory projection folder not found")
        return
    model_folder = os.path.join(train_cv_folder, model_name)
    if not os.path.exists(model_folder):
        print("Model folder not found")
        return
    training_folder = os.path.join(model_folder, 'training')
    if not os.path.exists(training_folder):
        print("Training folder not found")
        return
    traj_data_folder = os.path.join(model_folder, 'traj_data', system)
    if not os.path.exists(traj_data_folder):
        print("Trajectory data folder not found")
        return
    sensitivity_folder = os.path.join(model_folder, 'sensitivity_analysis')
    if not os.path.exists(sensitivity_folder):
        print("Sensitivity analysis folder not found")
        return

    # Show score and eigenfunctions if any
    if model_name in ['vae', 'ae', 'deep_tica']:
        score_path = os.path.join(training_folder, 'model_score.txt')
        if os.path.exists(score_path):
            show_score(score_path)
        else: print("Warning: Score file not found")
    if model_name == 'deep_tica':
        eig_path = os.path.join(training_folder, 'eigenvalues.txt')
        if os.path.exists(eig_path):
            show_eigenvalues(eig_path)
        else: print("Warning: Eigenvalues file not found")

    # Show evolution of CV for training data and given evaluation data
    if evaluation_traj_data is not None and evaluation_top_data is not None:
        proj_evaluation_data_paths = project_evaluation_data(evaluation_traj_data,
                                                            evaluation_top_data,
                                                            output_folder,
                                                            model_name)
    else:
        proj_evaluation_data_paths = []
    proj_training_data_path = os.path.join(traj_data_folder, 'projected_trajectory.csv')
    proj_train_plot = os.path.join(traj_data_folder, 'projected_trajectory.png')
    if os.path.exists(proj_training_data_path):
        create_time_evolution(proj_training_data_path, 
                              proj_evaluation_data_paths,
                              proj_train_plot)
    
    # Paths to images
    sensitivity_histogram = os.path.join(sensitivity_folder, 'sensitivity_histogram.png')
    trajectory = os.path.join(traj_data_folder, 'trajectory.png')
    loss = os.path.join(training_folder, 'loss.png')
    beta_loss = os.path.join(training_folder, 'vae_kl_reconstruction_loss.png')
    eigenvalues = os.path.join(training_folder, 'eigenvalues.png')
    fes = os.path.join(traj_projection_folder, model_name, 'fes', f'fes_{model_name}_1', 'fes.png')
    paths = [trajectory, loss, beta_loss, eigenvalues, proj_train_plot, fes, sensitivity_histogram]

    # Generate HTML image tags
    timestamp = int(time.time())
    images_html = [f'<img src="{path}?{timestamp}" style="width: 600px; margin-right: 10px;">' for path in paths if os.path.exists(path)]
    if not images_html:
        print("Warning: No images to display.")
        return

    # Display images
    display(HTML(''.join(images_html)))

Some examples from previous trainings:

/shared/work/pnavarro/projects/2025_DeepCartograph/LAT1/chain_B/deepCarto_analysis_iteration_4_dist_smallFeat_smallVAE_optimization/output_start_beta_0.000000001

/shared/work/pnavarro/projects/2025_DeepCartograph/LAT1/chain_B/deepCarto_analysis_iteration_4_dist_smallFeat_largeVAE_optimization/output_weight_decay_0.000000001



## Tests before production

Here we try different configurations, features and arquitectures to find good candidates for CV models.

The transition goes from 6IRS to 7DSQ.

### Configuration 1

- **Training set**: original GOdMD
- **Validation set**: original GOdMD
- **Evaluation set**: 2000-frame interpolated GOdMD with noise (no original frames)

In [ ]:
lag_time_array = [1,2,4,5]
for lag_time in lag_time_array:

    config_folder = f"{data_folder}/lat1/input"
    feature_list = ['distances']
    cv_models = ['deep_tica']

    # Main trajectory and topology  ---> to give as main (no augmentation) or seed (augmentation)
    version = "GOdMD_v1"
    training_replica = f"_raw_{lag_time}"
    GOdMD_input_path = f"{data_folder}/lat1/input/GOdMD_v1"
    traj_data = os.path.join(GOdMD_input_path, 'trajs','GOdMD_6IRS_7DSQ.dcd') 
    top_data = os.path.join(GOdMD_input_path, 'tops','GOdMD_6IRS_7DSQ.pdb')

    # (Optional) Endpoint equilibrations ---> supplementary data to project onto the CV
    # Restrained equilibrations at the endpoints of the reaction coordinate
    MD_input_path = f"{data_folder}/lat1/input/MD_equil"
    sup_traj_data = os.path.join(MD_input_path, 'trajs')
    sup_top_data = os.path.join(MD_input_path, 'tops')

    for features in feature_list:
        config_path = f"{config_folder}/{features}_config.yml"
        with open(config_path) as config_file:
            configuration = yaml.load(config_file, Loader = yaml.FullLoader)
        
        configuration['train_colvars']['common']['lag_time'] = lag_time
        
        # Output folder for the full workflow
        output_folder = f"{data_folder}/lat1/output/{version}/tests/{features}{training_replica}"

        # Clean output folder
        if os.path.exists(output_folder):
            shutil.rmtree(output_folder)

        # Run workflow
        deep_cartograph(
            configuration = configuration,
            trajectory_data = traj_data,
            topology_data = top_data,
            supplementary_traj_data = sup_traj_data,
            supplementary_top_data = sup_top_data,
            waypoints_data = sup_top_data,
            cvs = cv_models,
            dimension = 1,
            restart = True,
            output_folder = output_folder)

    from deep_cartograph.modules.common import read_list

    system_name = "GOdMD_6IRS_7DSQ"

    evaluation_folder = f"{data_folder}/lat1/input/GOdMD_v1_2000_noise_std0.2" #  GOdMD_v1_1000" 

    for features in feature_list:
        print("################################################################################################")
        print(f"Features: {features.upper()}")
        output_folder = f"{data_folder}/lat1/output/{version}/tests/{features}{training_replica}"
        path_full_set_features = os.path.join(output_folder, 'filter_features', 'all_features.txt')
        path_filtered_set_features = os.path.join(output_folder, 'filter_features', 'filtered_features.txt')
        full_set_features = read_list(path_full_set_features)
        filtered_set_features = read_list(path_filtered_set_features)
        print(f"Number of features in the full set: {len(full_set_features)}")
        print(f"Number of features in the filtered set: {len(filtered_set_features)}")
        for model in cv_models:
            print(f"Results for features: {features.upper()}")
            print(f"Results for model: {model.upper()}")
            print(f"Results for lag time: {lag_time} frames")
            show_results(output_folder, 
                        model_name = model, 
                        system = system_name,
                        evaluation_traj_data = os.path.join(evaluation_folder, 'trajs'),
                        evaluation_top_data = os.path.join(evaluation_folder, 'tops'))
        print("################################################################################################")

When increasing the lag time we reduce the number of pairs $x_t$, $x_{t+dt}$ that can be computed. For datasets that are already very small this can de-stabilize the training. For a given NN size, the optimal value will balance the filtering of fast modes with the number of samples available.

### Configuration 2

- **Training set**: balanced chimera GOdMD (original frames + MD endpoint samples)
- **Validation set**: balanced chimera GOdMD (original frames + MD endpoint samples)
- **Evaluation set**: 2000-frame interpolated GOdMD with noise (no original frames)

We can see that now the NN is able to clearly separate the enpoints 6IRS and 7DSQ MD samples in the CV space (which can be seen as a proxy for good generalization, at least near the endpoints). Additionally, with this larger dataset we are able to increase the lag time further without running into stability issues during training.

However, the CV is also capturing the artificial "breathing" motion introduced by the mixing of frames from MD and GOdMD, which is not desirable.

NOTE: in the evaluation data we are using there are not MD samples injected at the end points. Thus the transition takes place in a bigger proportion of the trajectory and the initial and final plateaus are shorter than in the training data when both training and evaluation are plotted considering the same simulation time (which is not true).

### Configuration 3

What we want is for the model to learn the main slow mode shown in the Chimeric trajectory (GOdMD + MD endpoints) but without overfitting to the artificial breathing motion. To achieve this, we can use the chimeric trajectory for training but validate on a different set that does not contain the MD endpoint frames. Note that this noisy validation set mantains the same time-scale as the chimeric trajectory used for training.

- **Training set**: balanced chimera GOdMD (original frames + MD endpoint samples)
- **Validation set**: 3 replicas of 140-frame interpolated GOdMD with noise 0.1 (no original frames)
- **Evaluation set**: 2000-frame interpolated GOdMD with noise (no original frames)

NOTE: Here it may be a good idea to increase the patience of the early stopping

Here we can see how the loss is lower in the training set -> we have learned the breathing between MD and godmd still, besides the problem with time scale change between training and validation

### Configuration 4

Increasing the noise in the validation set to make it more challenging.

- **Training set**: balanced chimera GOdMD (original frames + MD endpoint samples)
- **Validation set**: 3 replicas of 140-frame interpolated GOdMD with noise 0.2 (no original frames)
- **Evaluation set**: 2000-frame interpolated GOdMD with noise (no original frames)

### Configuration 5

Using a Chimera trajectory with more MD samples

- **Training set**: 540 samples - chimera GOdMD (original frames + MD endpoint samples) (more MD samples, less balanced)
- **Validation set**: 3 replicas of 140-frame interpolated GOdMD with noise 0.2 (no original frames)
- **Evaluation set**: 2000-frame interpolated GOdMD with noise (no original frames)

Still two ways we can improve this. Final strategy to construct Training and Validation for Deep TICA:


- Extend Chimeric trajectory for training
    - Include more MD endpoint-samples before and after the transition ($N_{MD} = 400$)
    - Total samples: $N_{godmd} + 2N_{MD}$ = 940
- Improve validation set: construct validation trajectories without MD samples but with the same time-scale as the training trajectory
    - Separate GOdMD into: state A ($N_A$) - transition ($N_T$) - state B ($N_B$) with $N_A + N_B + N_T = 140$ samples
    - Interpolate state A and state B to obtain $N_{MD} + N_A = 480$ and $N_{MD} + N_B = 430$ samples at each state
    - Join together: state A ($N_{MD} + N_A$) - transition ($N_T$) - state B ($N_{MD} + N_B$)
    - Interpolate to $N_{total}$ without original samples + noise! --> validate (if not good skip interpolation)

Like this we obtain a training set containing MD samples with a transition that appears more rare than in the initial GOdMD and a validation set that doesn't contain the aritificial MD to GOdMD breathing but just the slow transition with added noise. 

Both seem to capture the artificial oscillations in the training data due to the changes between simulation method. Next we try the same two trainings with an interpolated chimeric trajectory. This trajectory contains samples from the endpoints but the jumps between godmd and md have been significantly smoothed out. We also try to prove longer time scales

Last change in the training data. We make the endpoint plateaus symmetric. Meaning we include the same ratio of MD and godmd data so that the fluctuations correspond to the same time scale and we can ignore them fine tunning the lag time.

In [ ]:
lag_time_array = [1] # ,6,9,13,14,15,16] # Increase to see if oscillations disappear
for lag_time in lag_time_array:

    config_folder = f"{data_folder}/lat1/input"
    feature_list = ['distances']
    cv_models = ['deep_tica']

    # Main trajectory and topology  ---> to give as main (no augmentation) or seed (augmentation)
    version = "GOdMD_v1"
    training_replica = f"_chimera_{lag_time}_1"
    GOdMD_input_path = f"{data_folder}/lat1/input/GOdMD_v1_chimera"
    traj_data = os.path.join(GOdMD_input_path, 'trajs','GOdMD_6IRS_7DSQ_chimeric_890_symmetric_smoothed.dcd') 
    top_data = os.path.join(GOdMD_input_path, 'tops','GOdMD_6IRS_7DSQ_chimeric_890_symmetric_smoothed.pdb')

    # (Optional) Endpoint equilibrations ---> supplementary data to project onto the CV
    # Restrained equilibrations at the endpoints of the reaction coordinate
    MD_input_path = f"{data_folder}/lat1/input/MD_equil"
    sup_traj_data = os.path.join(MD_input_path, 'trajs')
    sup_top_data = os.path.join(MD_input_path, 'tops')

    # (Optional) Validation trajectories ---> validation data
    # (interpolated trajs with noise without the original frames)
    val_path = f"{data_folder}/lat1/input/GOdMD_v1_890"
    val_traj_data = f"{val_path}/trajs"
    val_top_data = f"{val_path}/tops"

    for features in feature_list:
        config_path = f"{config_folder}/{features}_config.yml"
        with open(config_path) as config_file:
            configuration = yaml.load(config_file, Loader = yaml.FullLoader)
        
        configuration['train_colvars']['common']['lag_time'] = lag_time
        
        configuration['train_colvars']['common']['architecture']['encoder']['dropout'] = [0.1, 0.1 , None]
        configuration['train_colvars']['common']['architecture']['decoder']['dropout'] = [None, 0.1, 0.1]
        
        configuration['train_colvars']['common']['training']['optimizer']['kwargs']['weight_decay'] = 0.0
        
        # Output folder for the full workflow
        output_folder = f"{data_folder}/lat1/output/{version}/tests/{features}{training_replica}"

        # Clean output folder
        if os.path.exists(output_folder):
            shutil.rmtree(output_folder)

        # Run workflow
        deep_cartograph(
            configuration = configuration,
            trajectory_data = traj_data,
            topology_data = top_data,
            validation_trajectory_data = val_traj_data,
            validation_topology_data = val_top_data,
            supplementary_traj_data = sup_traj_data,
            supplementary_top_data = sup_top_data,
            waypoints_data = sup_top_data,
            cvs = cv_models,
            dimension = 1,
            restart = True,
            output_folder = output_folder)

    from deep_cartograph.modules.common import read_list

    system_name = "GOdMD_6IRS_7DSQ_chimeric_890_symmetric_smoothed"

    evaluation_folder = f"{data_folder}/lat1/input/GOdMD_v1_chimera_890_noise_std0.2" #  GOdMD_v1_1000" 

    for features in feature_list:
        print("################################################################################################")
        print(f"Features: {features.upper()}")
        output_folder = f"{data_folder}/lat1/output/{version}/tests/{features}{training_replica}"
        path_full_set_features = os.path.join(output_folder, 'filter_features', 'all_features.txt')
        path_filtered_set_features = os.path.join(output_folder, 'filter_features', 'filtered_features.txt')
        full_set_features = read_list(path_full_set_features)
        filtered_set_features = read_list(path_filtered_set_features)
        print(f"Number of features in the full set: {len(full_set_features)}")
        print(f"Number of features in the filtered set: {len(filtered_set_features)}")
        for model in cv_models:
            print(f"Results for features: {features.upper()}")
            print(f"Results for model: {model.upper()}")
            print(f"Results for lag time: {lag_time} frames")
            show_results(output_folder, 
                        model_name = model, 
                        system = system_name,
                        evaluation_traj_data = os.path.join(evaluation_folder, 'trajs'),
                        evaluation_top_data = os.path.join(evaluation_folder, 'tops'))
        print("################################################################################################")

INFO:deep_cartograph:================
INFO:deep_cartograph:Analyze geometry
INFO:deep_cartograph:================
INFO:deep_cartograph:Elapsed time (Analyze geometry): 00 h 00 min 00 s
INFO:deep_cartograph:=======================
INFO:deep_cartograph:Trajectory Augmentation
INFO:deep_cartograph:=======================
INFO:deep_cartograph:Elapsed time (Trajectory Augmentation): 00 h 00 min 00 s
INFO:deep_cartograph:================
INFO:deep_cartograph:Compute features
INFO:deep_cartograph:================
/home/pnavarro/.conda/envs/deep_cartograph/lib/python3.10/site-packages/MDAnalysis/topology/PDBParser.py:350: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn("Element information is missing, elements attribute "
INFO:MDAnalysis.core.universe:attribute types has been guessed successfully.
INFO:MDAnalysis.core.universe:attri